#### Crash recovery in databases

In databases kunnen crashes voorkomen, vraag is dan of transacties op dat moment data corrupt (dirty, vies) hebben gemaakt. Om dit te voorkomen houden we een log bestand bij, dit log bestand staat veiligheidshalve op een andere schijf dan waar de database op staat. We definiëren het volgende 

$$
\begin{aligned}
&\text{INPUT}(X): && \text{copy block } X \text{ from disk to memory} \\
&\text{READ}(X,t): && \text{assign value of } X \text{ to variable } t; 
\text{ if necessary, this implicitly forces an } \text{INPUT}(X) \\
&\text{WRITE}(X,t): && \text{copy value of } t \text{ into } X \text{ in memory} \\
&\text{OUTPUT}(X): && \text{copy block } X \text{ from memory to disk} \\
& && \text{also called flushing } X
\end{aligned}
$$

Merk op dat disk dus staat voor static geheugen en memory voor werkgeheugen (RAM)

#### Voorbeeld (financial transaction):

Partial execution by failure:

$$
\begin{aligned}
&\text{INPUT(Account1);} \\
&\text{READ(Account1, v1);} \\
&v_1 := v_1 - 200; \\
&\text{WRITE(Account1, v1);} \\
&\text{OUTPUT(Account1);} \\
\\
&\text{>>> CRASH <<<} \\
\\
&\text{INPUT(Account2);} \\
&\text{READ(Account2, v2);} \\
&v_2 := v_2 + 200; \\
&\text{WRITE(Account2, v2);} \\
&\text{OUTPUT(Account2);}
\end{aligned}
$$

We zien hier dus dat 200 euro correct is afgeschreven, maar het wordt niet toe geschreven! Dit betekent dat de 200 euro dus verdwenen is.

Solution:

- Rollback transaction (undo partial changes)  
- As soon as possible: restart transaction

We hanteren de volgende regels voor het log bestand:

$$
\begin{aligned}
&\text{Old values of each data element } X \text{ must be written to the log file: } \langle T, X, \text{oldvalue} \rangle \\
&\text{T denotes the transaction identifier} \\
&\text{oldvalue is often called the before image of } X \\
&\text{Before doing } \text{OUTPUT}(X) \text{ the log record for this } X \text{ must be flushed} \\
&\text{The } \langle T,\text{ commit} \rangle \text{ record is written to the log after all database elements have been updated on disk}
\end{aligned}
$$

We kunnen dus alle data types die $T$ onder handen heeft genomen als valide beschouwen zodra we $\langle T,\text{ commit} \rangle $ zien staan

We definiëren en handhaven het volgende

$$
\begin{aligned}
&\text{Transaction } T \text{ will double the values of } A \text{ and } B \\[4pt]

&\text{D-A and D-B are the values of } A \text{ and } B \text{ on disk} \\[4pt]

&\text{M-A and M-B are the values of } A \text{ and } B \text{ in memory} \\[4pt]

& t \text{ is a variable in the programming environment} \\[4pt]

&\text{Log entries initially are collected in memory} \\[4pt]

&\text{Action FLUSH LOG forces the output of the log entries to the log file on disk}
\end{aligned}
$$

En bekijken dan het volgende

$$
\begin{array}{c|l|c|c|c|c|c|l}
\text{Step} & \text{Action} & t & \text{M-A} & \text{M-B} & \text{D-A} & \text{D-B} & \text{Log} \\
\hline
1 & \text{Start T} &  &  &  & 3 & 7 & \langle \text{START T} \rangle \\
2 & \text{READ(A,t)} & 3 & 3 &  & 3 & 7 &  \\
3 & t := t*2 & 6 & 3 &  & 3 & 7 &  \\
4 & \text{WRITE(A,t)} & 6 & 6 &  & 3 & 7 & \langle T, A, 3 \rangle \\
5 & \text{READ(B,t)} & 7 & 6 & 7 & 3 & 7 &  \\
6 & t := t*2 & 14 & 6 & 7 & 3 & 7 &  \\
7 & \text{WRITE(B,t)} & 14 & 6 & 14 & 3 & 7 & \langle T, B, 7 \rangle \\
8 & \text{FLUSH LOG} & 14 & 6 & 14 & 3 & 7 & \text{>>> log} \\
9 & \text{OUTPUT(A)} & 14 & 6 & 14 & 6 & 7 &  \\
10 & \text{OUTPUT(B)} & 14 & 6 & 14 & 6 & 14 &  \\
11 & \text{COMMIT T} & 14 & 6 & 14 & 6 & 14 & \langle \text{COMMIT T} \rangle \\
12 & \text{FLUSH LOG} & 14 & 6 & 14 & 6 & 14 & > \text{log}
\end{array}
$$

We zien dus dat de data op de disk $(D)$ Final is, en dat alle toepassing en bewerkingen in de RAM $(M)$ gebeurd.

We lezen dus vanuit de disk $(READ)$ data, wanneer we dit lezen zit het dus in de RAM. Dan voeren we iets uit met die data of veranderen we die data. Vervolgens schrijven we $(WRITE)$ de nieuwe data naar de RAM. Als laatst Schrijven we vanuit de RAM de nieuwe data op de drive $(OUTPUT)$.

stel nu dat er na stap 12 een crash komt, dan is transactie $T$ volledig goed uitgevoerd. Mocht er een crash komen bij een stap tussen 4 en 12, dan moeten we dus een undo uitvoeren!. We kijken dan in de Log kolom en zullen alle changes chronologisch terug moeten draaien.


$$
\begin{aligned}
&\textbf{Crash Recovery Rule} \\[4pt]

&\text{After a crash the disk may contain a mix of old and new values.} \\[4pt]

&\textbf{Committed transactions: REDO} \\
&\quad \text{Their effects must be guaranteed (durability), so updates are reapplied if necessary.} \\[6pt]

&\textbf{Uncommitted transactions: UNDO} \\
&\quad \text{Their effects must not appear in the database (atomicity), so updates are reverted.} \\[6pt]

&\textbf{Goal:} \\
&\quad \text{Database state = as if all committed transactions executed} \\
&\quad \text{and all uncommitted transactions never happened.}
\end{aligned}
$$

#### Dirty data

$$
\begin{aligned}
&\textbf{Reads-from (RF)}: \\
&\quad T_j \ RF \ T_i \ \text{if there exists } X \text{ such that } 
W_i(X) \text{ is the last write on } X \text{ before } R_j(X). \\[6pt]

&\textbf{Dirty read}: \\
&\quad \text{If } T_j \ RF \ T_i \text{ and } T_i \text{ has not committed yet,} \\
&\quad \text{then the read by } T_j \text{ is called a dirty read.} \\[6pt]

&\textbf{Recoverable schedule (RC)}: \\
&\quad \text{A schedule is recoverable if for every pair of transactions:} \\
&\quad T_j \ RF \ T_i \ \Rightarrow\ \text{COMMIT}_i < \text{COMMIT}_j \\[6pt]

&\textbf{Meaning}: \\
&\quad < \text{ denotes time order.} \\[4pt]
&\quad \text{A schedule is RC if it does not contain dirty reads.}
\end{aligned}
$$

We willen dus geen dirty data in onze database, dit zorgt er namelijk voor dat in het geval van een crash dat de database niet recoverable is!

- **Definition:** A schedule is **strict (ST)** if:

$$
W_i(X) < O_j(X) \;\Rightarrow\; 
COMMIT_i < O_j(X) \;\text{ or }\; ABORT_i < O_j(X)
$$

for each read or write operation $O_j$ by transaction $T_j$ on item $X$.

- **Implementation of strictness with 2PL:**  
  Strictness can be combined with **Two-Phase Locking (2PL)**:  
  a transaction must hold all its **write locks** until **COMMIT** or **ABORT**.

### Recovery strategieën

#### UNDO logging

$$
\textbf{Idee: } \text{Wijzigingen kunnen al op disk staan voordat de transactie commit.}
$$

Bij een crash:

- Gecommitte transacties → **niets doen**
- Niet-gecommitte transacties → **UNDO**

$$
\text{Recovery: } \text{Undo alle incomplete transacties}
$$

#### REDO logging

$$
\textbf{Idee: } \text{Wijzigingen worden pas later naar disk geschreven.}
$$

Bij een crash:

- Gecommitte transacties → **REDO**
- Niet-gecommitte transacties → **negeren**

$$
\text{Recovery: } \text{Redo alle gecommitte transacties}
$$


#### UNDO/REDO logging

$$
\textbf{Idee: } \text{Combinatie van beide methodes.}
$$

Bij een crash:

- Gecommitte transacties → **REDO**
- Niet-gecommitte transacties → **UNDO**

$$
\text{Recovery policy:}
$$

$$
\begin{aligned}
&\text{Redo committed transactions (earliest-first)} \\
&\text{Undo incomplete transactions (latest-first)}
\end{aligned}
$$

We kunnen ons voorstellen dat wanneer we een hele grote log bestand hebben in een grote database, we bij een crash niet door het hele log bestand willen ploegen. We gebruiken daarom checkpoints, tot de laatste checkpoint is de database volledig en hoeven we het alleen tot op dat punt terug te zetten. Wanneer er dit staat in een log bestand:

$$
    \langle \text{START CKPT} (T_1, T_2) \rangle
$$

Zegt dit dat we de checkpoint gestart is terwijl $T_1$ en $T_2$ nog actief waren, die zijn dus nog niet gecommit. Bekijk het voorbeeld hieronder waar we $\textbf{UNDO}$ toepassen

### UNDO Log

$$
\begin{aligned}
&\langle START\ T1 \rangle \\
&\langle T1,\ B,\ 10 \rangle \\
&\langle COMMIT\ T1 \rangle \\
&\langle START\ T3 \rangle \\
&\langle START\ T2 \rangle \\
&\langle T2,\ A,\ 5 \rangle \\
&\langle T3,\ C,\ 10 \rangle \\
&\langle START\ CKPT\ (T2, T3) \rangle \\
&\langle T2,\ D,\ 15 \rangle \\
&\langle START\ T4 \rangle \\
&\langle START\ T5 \rangle \\
&\langle T4,\ F,\ 25 \rangle \\
&\langle T5,\ H,\ 12 \rangle \\
&\langle COMMIT\ T2 \rangle
\end{aligned}
$$

Crash occurs after:

$$
\langle COMMIT\ T2 \rangle
$$

### Recovery analysis

- Scan the log **backwards** from the last entry.

- The first checkpoint entry encountered is:

$$
\langle START\ CKPT\ (T2, T3) \rangle
$$

- This means that updates by $T_3$ **before the checkpoint** may still exist and must be considered.

- Therefore we scan further back until:

$$
\langle START\ T3 \rangle
$$

- Transactions that **did not commit before the crash**:

$$
T_3,\ T_4,\ T_5
$$

- Hence we must:

$$
\text{UNDO updates of } T_3,\ T_4,\ T_5
$$

#### REDO logging

$$
\begin{aligned}
&\langle START\ T1 \rangle \\
&\langle T1,\ B,\ 10 \rangle \\
&\langle COMMIT\ T1 \rangle \\
&\langle START\ T3 \rangle \\
&\langle START\ T2 \rangle \\
&\langle T2,\ A,\ 5 \rangle \\
&\langle T3,\ C,\ 10 \rangle \\
&\langle START\ CKPT\ (T2, T3) \rangle \\
&\langle T2,\ D,\ 15 \rangle \\
&\langle START\ T4 \rangle \\
&\langle START\ T5 \rangle \\
&\langle T4,\ F,\ 25 \rangle \\
&\langle T5,\ H,\ 12 \rangle \\
&\langle COMMIT\ T2 \rangle
\end{aligned}
$$

Crash occurs after:

$$
\langle COMMIT\ T2 \rangle
$$

#### Recovery (REDO logging)

- Scan the log **backwards** from the last entry.

- The first checkpoint entry encountered is

$$
\langle START\ CKPT\ (T2, T3) \rangle
$$

- Therefore we must scan **all the way back to the beginning of the log**, unless we encounter

$$
\langle END\ CKPT \rangle
$$

first.

- In this case we **do not encounter** an $\langle END\ CKPT \rangle$, so we scan to the beginning.

- Transactions that **committed before the crash**:

$$
T_1,\ T_2
$$

- Hence we must:

$$
\text{REDO updates of } T_1 \text{ and } T_2
$$

#### UNDO/REDO logging

**Combined UNDO/REDO logging**

$$
\begin{aligned}
&\langle START\ T1 \rangle \\
&\langle T1,\ B,\ 10,\ 11 \rangle \\
&\langle COMMIT\ T1 \rangle \\
&\langle START\ T3 \rangle \\
&\langle START\ T2 \rangle \\
&\langle T2,\ A,\ 5,\ 6 \rangle \\
&\langle T3,\ C,\ 10,\ 11 \rangle \\
&\langle START\ CKPT\ (T2, T3) \rangle \\
&\langle T2,\ D,\ 15,\ 16 \rangle \\
&\langle START\ T4 \rangle \\
&\langle START\ T5 \rangle \\
&\langle T4,\ F,\ 25,\ 26 \rangle \\
&\langle T5,\ H,\ 12,\ 13 \rangle \\
&\langle COMMIT\ T2 \rangle
\end{aligned}
$$

Crash occurs after:

$$
\langle COMMIT\ T2 \rangle
$$

### Recovery (UNDO/REDO logging)

- With **UNDO/REDO logging**:
  
$$
\text{Committed transactions } \rightarrow \text{ REDO}
$$

$$
\text{Uncommitted transactions } \rightarrow \text{ UNDO}
$$

- Scan the log **backwards** from the last entry.

- The first checkpoint entry encountered is

$$
\langle START\ CKPT\ (T2, T3) \rangle
$$

- Therefore we must scan **all the way back to the beginning of the log** unless we encounter

$$
\langle END\ CKPT \rangle
$$

- In this case we **do not encounter** an $\langle END\ CKPT \rangle$, so we scan to the beginning.

- Transactions that **committed before the crash**:

$$
T_1,\ T_2
$$

$$
\Rightarrow \text{REDO updates of } T_1,\ T_2
$$

- Transactions that **did not commit before the crash**:

$$
T_3,\ T_4,\ T_5
$$

$$
\Rightarrow \text{UNDO updates of } T_3,\ T_4,\ T_5
$$